# **Bridge Condition Rating Predcition Python Code**

# **Install & Mount**

In [2]:
!pip install -U pip
!pip install autogluon openpyxl

import os, time, numpy as np, pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from autogluon.tabular import TabularPredictor

from google.colab import drive
drive.mount('/content/drive')

import torch
HAS_GPU = torch.cuda.is_available()
print("GPU available:", HAS_GPU)

Mounted at /content/drive
GPU available: True


# **Settings (edit years/paths here)**

In [3]:
# Paths
DRIVE_ROOT = '/content/drive/My Drive'
DATA_DIR   = f'{DRIVE_ROOT}/BridgeResearchProject/NBIDataCollected'
OUT_DIR    = f'{DRIVE_ROOT}/BridgeResearchProject/PredictModels'
os.makedirs(OUT_DIR, exist_ok=True)

# Years
START_YEAR = 2024
END_YEAR   = 1992   # inclusive
ALL_DESC   = list(range(START_YEAR, END_YEAR-1, -1))  # [2024, 2023, ..., 1992]

# Label / filtering
LABEL = 'Deck_Rate'
CLASS_KEEP = [4,5,6,7,8,9]

# Resume behavior: skip an iteration if this CSV already exists
def report_path(max_y, min_y):
    return os.path.join(OUT_DIR, f"model_report_DeckRate_XGB_{max_y}_{min_y}.csv")

# Cache for loaded Excel (avoids re-reading same year repeatedly)
YEAR_CACHE = {}


# **Features and Data Frame**

In [ ]:
def _read_year_df(year: int, cols: list) -> pd.DataFrame | None:
    """Read California{year}.xlsx with selected columns; cached."""
    if year in YEAR_CACHE:
        return YEAR_CACHE[year]
    fp = os.path.join(DATA_DIR, f'California{year}.xlsx')
    if not os.path.exists(fp):
        print(f'File not found: California{year}.xlsx')
        YEAR_CACHE[year] = None
        return None
    df = pd.read_excel(fp, usecols=lambda c: c in cols)
    df['year'] = year
    YEAR_CACHE[year] = df
    return df

def assemble_df(years_desc: list[int]) -> pd.DataFrame:
    COLS = [
        'DATE_OF_INSPECT_090','YEAR_BUILT_027','YEAR_RECONSTRUCTED_106',
        'ADT_029','YEAR_ADT_030','UNDWATER_LAST_DATE_093B','FUTURE_ADT_114',
        'RIGHT_CURB_MT_050B','LEFT_CURB_MT_050A','STRUCTURE_LEN_MT_049',
        'DECK_WIDTH_MT_052','PERCENT_ADT_TRUCK_109','TRAFFIC_LANES_ON_028A',
        'MAIN_UNIT_SPANS_045','MAX_SPAN_LEN_MT_048','OPERATING_RATING_064',
        'HIGHWAY_DISTRICT_002','DESIGN_LOAD_031','STRUCTURE_KIND_043A',
        'STRUCTURE_TYPE_043B','APPR_KIND_044A','APPR_TYPE_044B',
        'DECK_GEOMETRY_EVAL_068','DECK_STRUCTURE_TYPE_107','SURFACE_TYPE_108A',
        'DECK_COND_058'
    ]
    frames = []

    for y in years_desc:
        df_y = _read_year_df(y, COLS)
        if df_y is not None:
            frames.append(df_y)
    if not frames:
        raise ValueError("No data found for the requested years.")
    df = pd.concat(frames, ignore_index=True)

    # ---- Features (compact) ----
    df['DATE_OF_INSPECT_090'] = pd.to_numeric(df['DATE_OF_INSPECT_090'], errors='coerce')
    df['YEAR_BUILT_027']      = pd.to_numeric(df['YEAR_BUILT_027'], errors='coerce')
    insp_two = (df['DATE_OF_INSPECT_090'] % 100).fillna(0)
    df['Age'] = 2000 + insp_two - df['YEAR_BUILT_027'].fillna(df['YEAR_BUILT_027'].median())

    df['YEAR_RECONSTRUCTED_106'] = pd.to_numeric(df['YEAR_RECONSTRUCTED_106'], errors='coerce').fillna(0)
    df['Reconstructed'] = (df['YEAR_RECONSTRUCTED_106'] != 0).astype('int8')

    yi = pd.to_datetime(df['UNDWATER_LAST_DATE_093B'], errors='coerce').dt.year.fillna(df['year'])
    df['_YI_'] = (yi % 100).astype(float)

    A = pd.to_numeric(df['ADT_029'], errors='coerce')
    F = pd.to_numeric(df['FUTURE_ADT_114'], errors='coerce')
    Y = pd.to_numeric(df['YEAR_ADT_030'], errors='coerce')
    num = (F - A)
    den = (F - Y)
    with np.errstate(divide='ignore', invalid='ignore'):
        frac = np.where((den != 0) & np.isfinite(den), num/den, np.nan)
        df['ADT'] = frac * (df['_YI_'] - Y) + A

    df['Curb_Width'] = pd.to_numeric(df['LEFT_CURB_MT_050A'], errors='coerce').fillna(0) + \
                       pd.to_numeric(df['RIGHT_CURB_MT_050B'], errors='coerce').fillna(0)
    df['Deck_Area']  = pd.to_numeric(df['STRUCTURE_LEN_MT_049'], errors='coerce').fillna(0) * \
                       pd.to_numeric(df['DECK_WIDTH_MT_052'], errors='coerce').fillna(0)
    df['Deck_Rate']  = pd.to_numeric(df['DECK_COND_058'], errors='coerce').fillna(0)

    df.rename(columns={
        'PERCENT_ADT_TRUCK_109':'ADTT',
        'TRAFFIC_LANES_ON_028A':'Lanes_On',
        'MAIN_UNIT_SPANS_045':'Number_Spans_Main',
        'MAX_SPAN_LEN_MT_048':'Length_Max_Span',
        'OPERATING_RATING_064':'Operating_Rating',
        'HIGHWAY_DISTRICT_002':'Highway_District',
        'DESIGN_LOAD_031':'Design_Load',
        'STRUCTURE_KIND_043A':'Main_Material',
        'STRUCTURE_TYPE_043B':'Main_Design',
        'APPR_KIND_044A':'Spans_Material',
        'APPR_TYPE_044B':'Spans_Design',
        'DECK_GEOMETRY_EVAL_068':'Deck_Geometry',
        'DECK_STRUCTURE_TYPE_107':'Deck_Type',
        'SURFACE_TYPE_108A':'Wearing_Surface',
    }, inplace=True)

    ordered = ['Age','ADT','ADTT','Lanes_On','Number_Spans_Main','Length_Max_Span',
               'Curb_Width','Deck_Area','Operating_Rating','Highway_District',
               'Design_Load','Reconstructed','Main_Material','Main_Design',
               'Spans_Material','Spans_Design','Deck_Geometry','Deck_Type',
               'Wearing_Surface','Deck_Rate']
    df = df[ordered].copy()

    for c in ['Highway_District','Design_Load','Reconstructed','Main_Material',
              'Main_Design','Spans_Material','Spans_Design','Deck_Geometry',
              'Deck_Type','Wearing_Surface']:
        df[c] = df[c].astype('category')

    df = df[df['Deck_Rate'].isin(CLASS_KEEP)].copy()
    return df

def train_and_report(final_df: pd.DataFrame, max_y: int, min_y: int):
    """Train AutoGluon, save predictor + combined (classification report + confusion matrix)."""
    # Stable 90/10 split
    test = final_df.sample(frac=0.10, random_state=30)
    train = final_df.drop(test.index)

    save_stub  = f"DeckRate_XGB_{max_y}_{min_y}"
    save_path  = os.path.join(OUT_DIR, save_stub)
    os.makedirs(save_path, exist_ok=True)

    # Choose a single model (XGBoost here)
    xgb_hps = {
        'XGB': {'tree_method': 'gpu_hist' if HAS_GPU else 'hist'}
    }

    predictor = TabularPredictor(label='Deck_Rate', path=save_path).fit(
        train,
        hyperparameters=xgb_hps,
        holdout_frac=0.2
    )

    # Evaluate
    y_true = test['Deck_Rate']
    y_pred = predictor.predict(test)

    report = classification_report(y_true, y_pred, output_dict=True)
    cm = confusion_matrix(y_true, y_pred, labels=predictor.class_labels)

    # Save classification report + confusion matrix
    report_df = pd.DataFrame(report).transpose()
    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual_{c}" for c in predictor.class_labels],
        columns=[f"Pred_{c}" for c in predictor.class_labels]
    )
    combined = pd.concat([report_df, pd.DataFrame([[]]), cm_df], axis=0)
    out_csv = report_path(max_y, min_y)
    combined.to_csv(out_csv)
    print(f"✅ Saved: {out_csv}")
    report_df = pd.DataFrame(report).transpose()
    print(report_df)



# **Cumulative loop with Training + Evaluation**

In [ ]:
start_time = time.time()

for k in range(1, len(ALL_DESC)+1):
    years_now = ALL_DESC[:k]             # cumulative: [2024], [2024,2023], ...
    max_y, min_y = years_now[0], years_now[-1]
    out_csv = report_path(max_y, min_y)

    if os.path.exists(out_csv):
        print(f"⏭️  Skipping {max_y}-{min_y} (already exists)")
        continue

    print(f"\n=== Running {max_y}-{min_y} (years={years_now}) ===")
    t0 = time.time()
    df = assemble_df(years_now)
    print(f"Rows: {len(df)} | classes: {sorted(df['Deck_Rate'].unique().tolist())}")
    print(df)
    train_and_report(df, max_y, min_y)
    print(f"⏱️ Took {(time.time()-t0)/60:.1f} min")

# 🔒 Ensure the final comprehensive run exists (2024–1992)
final_csv = report_path(START_YEAR, END_YEAR)
if not os.path.exists(final_csv):
    print(f"\n🔁 Final report {START_YEAR}-{END_YEAR} is creating ...")
    df_all = assemble_df(ALL_DESC)
    print(df_all['Deck_Rate'].value_counts())
    train_and_report(df_all, START_YEAR, END_YEAR)
else:
    print(f"\n✅ Final report already present: {final_csv}")

print(f"\nTotal elapsed: {(time.time()-start_time)/60:.2f} mins")


⏭️  Skipping 2024-2024 (already exists)

=== Running 2024-2023 (years=[2024, 2023]) ===


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct  2 10:42:05 UTC 2025
CPU Count:          2
Memory Avail:       10.39 GB / 12.67 GB (82.0%)
Disk Space Avail:   37.29 GB / 100.00 GB (37.3%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme' : New in v1.4: Massively better than 'best' on datasets <30000 samples by using new models meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitra, and TabM. Absolute best accuracy. Requires a GPU. Recommended 64 GB CPU memory and 32+ GB GPU memory.
	presets='best'    : Maximize accuracy. Recommended for most users. Use in competitions and

Rows: 43761 | classes: [4.0, 5.0, 6.0, 7.0, 8.0, 9.0]
       Age           ADT  ADTT  Lanes_On  Number_Spans_Main  Length_Max_Span  \
0       81  18523.156345  29.0         4                  8            192.0   
1       74    255.104935   2.0         2                  1             10.4   
2       38    100.000000  20.0         2                  5             18.9   
3       75    256.409529   2.0         2                  4              6.9   
4       75    255.104935   2.0         4                  4              6.9   
...    ...           ...   ...       ...                ...              ...   
51654   36   7139.057971  12.0         2                  1              9.1   
51655   36   7139.057971  12.0         2                  2              9.1   
51656   36   6706.207729  12.0         2                  2             10.7   
51662   49   2726.550543   5.0         2                  2             24.4   
51665   81     10.994030   NaN         2                  1       

Beginning AutoGluon training ...
AutoGluon will save models to "/content/drive/My Drive/BridgeResearchProject/PredictModels/DeckRate_XGB_2024_2023"
Train Data Rows:    39385
Train Data Columns: 19
Label Column:       Deck_Rate
AutoGluon infers your prediction problem is: 'multiclass' (because dtype of label-column == float, but few unique label-values observed and label-values can be converted to int).
	6 unique label values:  [np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(9.0), np.float64(4.0), np.float64(8.0)]
	If 'multiclass' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       multiclass
Preprocessing data ...
Train Data Class Count: 6
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    10658.69 MB
	Train Data (Original)  Me

✅ Saved: /content/drive/My Drive/BridgeResearchProject/PredictModels/model_report_DeckRate_XGB_2024_2023.csv
              precision    recall  f1-score      support
4.0            0.802469  0.680628  0.736544   191.000000
5.0            0.820763  0.775356  0.797414  1193.000000
6.0            0.780876  0.678201  0.725926   289.000000
7.0            0.893752  0.938473  0.915567  2698.000000
8.0            1.000000  0.600000  0.750000     5.000000
accuracy       0.865174  0.865174  0.865174     0.865174
macro avg      0.859572  0.734532  0.785090  4376.000000
weighted avg   0.862536  0.865174  0.862828  4376.000000
⏱️ Took 3.3 min

✅ Final report already present: /content/drive/My Drive/BridgeResearchProject/PredictModels/model_report_DeckRate_XGB_2024_2023.csv

Total elapsed: 3.33 mins




```
# This is formatted as code
```

# **Evaluate 2025 against all saved models**

In [4]:
def _read_test_year_df(year: int, cols: list) -> pd.DataFrame | None:
    """Read California{year}.xlsx with selected columns; cached."""
    fp = os.path.join(DATA_DIR, f'California{TEST_YEAR}.xlsx')
    if not os.path.exists(fp):
        print(f'File not found: California{year}.xlsx')
        return None
    df = pd.read_excel(fp, usecols=lambda c: c in cols)
    df['year'] = TEST_YEAR
    return df
def assemble_df_test(years_desc: list[int]) -> pd.DataFrame:
    COLS = [
        'DATE_OF_INSPECT_090','YEAR_BUILT_027','YEAR_RECONSTRUCTED_106',
        'ADT_029','YEAR_ADT_030','UNDWATER_LAST_DATE_093B','FUTURE_ADT_114',
        'RIGHT_CURB_MT_050B','LEFT_CURB_MT_050A','STRUCTURE_LEN_MT_049',
        'DECK_WIDTH_MT_052','PERCENT_ADT_TRUCK_109','TRAFFIC_LANES_ON_028A',
        'MAIN_UNIT_SPANS_045','MAX_SPAN_LEN_MT_048','OPERATING_RATING_064',
        'HIGHWAY_DISTRICT_002','DESIGN_LOAD_031','STRUCTURE_KIND_043A',
        'STRUCTURE_TYPE_043B','APPR_KIND_044A','APPR_TYPE_044B',
        'DECK_GEOMETRY_EVAL_068','DECK_STRUCTURE_TYPE_107','SURFACE_TYPE_108A',
        'DECK_COND_058'
    ]
    frames = []
    for y in years_desc:
        df_y = _read_test_year_df(y, COLS)
        if df_y is not None:
            frames.append(df_y)
    if not frames:
        raise ValueError("No data found for the requested years.")
    df = pd.concat(frames, ignore_index=True)

    # ---- Features (compact) ----
    df['DATE_OF_INSPECT_090'] = pd.to_numeric(df['DATE_OF_INSPECT_090'], errors='coerce')
    df['YEAR_BUILT_027']      = pd.to_numeric(df['YEAR_BUILT_027'], errors='coerce')
    insp_two = (df['DATE_OF_INSPECT_090'] % 100).fillna(0)
    df['Age'] = 2000 + insp_two - df['YEAR_BUILT_027'].fillna(df['YEAR_BUILT_027'].median())

    df['YEAR_RECONSTRUCTED_106'] = pd.to_numeric(df['YEAR_RECONSTRUCTED_106'], errors='coerce').fillna(0)
    df['Reconstructed'] = (df['YEAR_RECONSTRUCTED_106'] != 0).astype('int8')

    yi = pd.to_datetime(df['UNDWATER_LAST_DATE_093B'], errors='coerce').dt.year.fillna(df['year'])
    df['_YI_'] = (yi % 100).astype(float)

    A = pd.to_numeric(df['ADT_029'], errors='coerce')
    F = pd.to_numeric(df['FUTURE_ADT_114'], errors='coerce')
    Y = pd.to_numeric(df['YEAR_ADT_030'], errors='coerce')
    num = (F - A)
    den = (F - Y)
    with np.errstate(divide='ignore', invalid='ignore'):
        frac = np.where((den != 0) & np.isfinite(den), num/den, np.nan)
        df['ADT'] = frac * (df['_YI_'] - Y) + A

    df['Curb_Width'] = pd.to_numeric(df['LEFT_CURB_MT_050A'], errors='coerce').fillna(0) + \
                       pd.to_numeric(df['RIGHT_CURB_MT_050B'], errors='coerce').fillna(0)
    df['Deck_Area']  = pd.to_numeric(df['STRUCTURE_LEN_MT_049'], errors='coerce').fillna(0) * \
                       pd.to_numeric(df['DECK_WIDTH_MT_052'], errors='coerce').fillna(0)
    df['Deck_Rate']  = pd.to_numeric(df['DECK_COND_058'], errors='coerce').fillna(0)

    df.rename(columns={
        'PERCENT_ADT_TRUCK_109':'ADTT',
        'TRAFFIC_LANES_ON_028A':'Lanes_On',
        'MAIN_UNIT_SPANS_045':'Number_Spans_Main',
        'MAX_SPAN_LEN_MT_048':'Length_Max_Span',
        'OPERATING_RATING_064':'Operating_Rating',
        'HIGHWAY_DISTRICT_002':'Highway_District',
        'DESIGN_LOAD_031':'Design_Load',
        'STRUCTURE_KIND_043A':'Main_Material',
        'STRUCTURE_TYPE_043B':'Main_Design',
        'APPR_KIND_044A':'Spans_Material',
        'APPR_TYPE_044B':'Spans_Design',
        'DECK_GEOMETRY_EVAL_068':'Deck_Geometry',
        'DECK_STRUCTURE_TYPE_107':'Deck_Type',
        'SURFACE_TYPE_108A':'Wearing_Surface',
    }, inplace=True)

    ordered = ['Age','ADT','ADTT','Lanes_On','Number_Spans_Main','Length_Max_Span',
               'Curb_Width','Deck_Area','Operating_Rating','Highway_District',
               'Design_Load','Reconstructed','Main_Material','Main_Design',
               'Spans_Material','Spans_Design','Deck_Geometry','Deck_Type',
               'Wearing_Surface','Deck_Rate']
    df = df[ordered].copy()

    for c in ['Highway_District','Design_Load','Reconstructed','Main_Material',
              'Main_Design','Spans_Material','Spans_Design','Deck_Geometry',
              'Deck_Type','Wearing_Surface']:
        df[c] = df[c].astype('category')

    df = df[df['Deck_Rate'].isin(CLASS_KEEP)].copy()
    return df

In [5]:
from sklearn.metrics import accuracy_score, confusion_matrix,f1_score
import json

TEST_YEAR = 2023

# 1) Load and prepare the test dataframe (X, y)
try:
    df_test = assemble_df_test([TEST_YEAR])  # reuses your feature engineering + filters
    print(df_test['Deck_Rate'].unique())
    print(df_test['Deck_Rate'].value_counts())
except Exception as e:
    raise RuntimeError(
        f"Could not assemble the {TEST_YEAR} dataframe. "
        f"Make sure {DATA_DIR}/California{TEST_YEAR}.xlsx exists and has required columns."
    ) from e

if df_test.empty:
    raise RuntimeError(f"No rows available for {TEST_YEAR} after filtering CLASS_KEEP={CLASS_KEEP}.")

y_true = df_test['Deck_Rate'].copy()
X_test = df_test.drop(columns=['Deck_Rate']).copy()

[5. 6. 7. 8. 4. 9.]
Deck_Rate
7.0    13564
5.0     5786
6.0     1437
4.0     1038
8.0       50
9.0        6
Name: count, dtype: int64


In [6]:
# 2) Find all saved model directories
model_dirs = [
    d for d in os.listdir(OUT_DIR)
    if d.startswith('DeckRate_XGB_') and os.path.isdir(os.path.join(OUT_DIR, d))
]
model_dirs = sorted(model_dirs)  # nice, stable order
print(model_dirs)

if not model_dirs:
    raise RuntimeError(f"No saved models found in {OUT_DIR}. Train cells must run first.")

summary_rows = []

['DeckRate_XGB_2024_1992', 'DeckRate_XGB_2024_1993', 'DeckRate_XGB_2024_1994', 'DeckRate_XGB_2024_1995', 'DeckRate_XGB_2024_1996', 'DeckRate_XGB_2024_1997', 'DeckRate_XGB_2024_1998', 'DeckRate_XGB_2024_1999', 'DeckRate_XGB_2024_2000', 'DeckRate_XGB_2024_2001', 'DeckRate_XGB_2024_2002', 'DeckRate_XGB_2024_2003', 'DeckRate_XGB_2024_2004', 'DeckRate_XGB_2024_2005', 'DeckRate_XGB_2024_2006', 'DeckRate_XGB_2024_2007', 'DeckRate_XGB_2024_2008', 'DeckRate_XGB_2024_2009', 'DeckRate_XGB_2024_2010', 'DeckRate_XGB_2024_2011', 'DeckRate_XGB_2024_2012', 'DeckRate_XGB_2024_2013', 'DeckRate_XGB_2024_2014', 'DeckRate_XGB_2024_2015', 'DeckRate_XGB_2024_2016', 'DeckRate_XGB_2024_2017', 'DeckRate_XGB_2024_2018', 'DeckRate_XGB_2024_2019', 'DeckRate_XGB_2024_2020', 'DeckRate_XGB_2024_2021', 'DeckRate_XGB_2024_2022', 'DeckRate_XGB_2024_2023', 'DeckRate_XGB_2024_2024']


In [7]:
# 3) Evaluate each model on the teat year holdout
# model_dirs = ['DeckRate_XGB_2024_1992', 'DeckRate_XGB_2024_1993', 'DeckRate_XGB_2024_1994',
#               'DeckRate_XGB_2024_1995', 'DeckRate_XGB_2024_1996', 'DeckRate_XGB_2024_1997',
#               'DeckRate_XGB_2024_1998', 'DeckRate_XGB_2024_1999', 'DeckRate_XGB_2024_2000',
#               'DeckRate_XGB_2024_2001', 'DeckRate_XGB_2024_2002', 'DeckRate_XGB_2024_2003',
#               'DeckRate_XGB_2024_2004', 'DeckRate_XGB_2024_2005', 'DeckRate_XGB_2024_2006',
#               'DeckRate_XGB_2024_2007', 'DeckRate_XGB_2024_2008', 'DeckRate_XGB_2024_2009',
#               'DeckRate_XGB_2024_2010', 'DeckRate_XGB_2024_2011', 'DeckRate_XGB_2024_2012',
#               'DeckRate_XGB_2024_2013', 'DeckRate_XGB_2024_2014', 'DeckRate_XGB_2024_2015',
#               'DeckRate_XGB_2024_2016', 'DeckRate_XGB_2024_2017', 'DeckRate_XGB_2024_2018',
#               'DeckRate_XGB_2024_2019', 'DeckRate_XGB_2024_2020', 'DeckRate_XGB_2024_2021',
#               'DeckRate_XGB_2024_2022','DeckRate_XGB_2024_2023','DeckRate_XGB_2024_2024']
model_dirs = ['DeckRate_XGB_2024_2024']
for md in model_dirs:
  start_time = time.time()
  model_path = os.path.join(OUT_DIR, md)
  try:
      predictor = TabularPredictor.load(model_path)
      print(model_path)
  except Exception as e:
      print(f"⚠️ Skipping {md}: could not load predictor ({e})")
      continue

  # Predict on test year features
  y_pred = predictor.predict(X_test)

  # Overall accuracy and f1
  acc = accuracy_score(y_true, y_pred)
  f1_macro    = f1_score(y_true, y_pred, average='macro', zero_division=0)
  f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)


  # Confusion matrix (fixed order = predictor.class_labels)
  labels = list(predictor.class_labels)  # preserves training label order
  cm = confusion_matrix(y_true, y_pred, labels=labels)

  # Save a compact CSV combining accuracy and CM for this model
  cm_df = pd.DataFrame(cm, index=[f"Actual_{c}" for c in labels],
                          columns=[f"Pred_{c}" for c in labels])
  # Put accuracy on a single-row DataFrame and concat above the CM
  # acc_df = pd.DataFrame({"metric": ["accuracy"], "value": [acc]}).set_index("metric")
  # out_csv = os.path.join(OUT_DIR, f"eval_{TEST_YEAR}_{md}.csv")
  # combined = pd.concat([acc_df, pd.DataFrame([[]]), cm_df], axis=0)
  # combined.to_csv(out_csv)

  # Keep a short summary row for on-screen review
  # md is like DeckRate_XGB_2024_2019 — split it for readability
  try:
      _, _, max_y, min_y = md.split("_")
      model_span = f"{max_y}-{min_y}"
  except Exception:
      model_span = md.replace("DeckRate_XGB_", "")
  summary_rows.append({"model_span": model_span, "rows_2025": len(X_test), "accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted})
  print(f"\nTotal elapsed: {(time.time()-start_time)/60:.2f} mins, {model_span}, {acc},{f1_weighted}")

# 4) Show a  summary table; highest accuracy first
summary_df = pd.DataFrame(summary_rows).sort_values("accuracy", ascending=False)
print("✅ 2025 evaluation across saved models (higher is better):")
display(summary_df)

# (Optional) Also write a single summary CSV
summary_csv = os.path.join(OUT_DIR, f"eval_{TEST_YEAR}_SUMMARY.csv")
summary_df.to_csv(summary_csv, index=False)
# print(f"📁 Wrote per-model CSVs and summary to:\n  - {summary_csv}")

/content/drive/My Drive/BridgeResearchProject/PredictModels/DeckRate_XGB_2024_2024

Total elapsed: 0.14 mins, 2024-2024, 0.7389973035967278,0.7122959389252337
✅ 2025 evaluation across saved models (higher is better):


,model_span,rows_2025,accuracy,f1_macro,f1_weighted
0,2024-2024,21881,0.738997,0.471391,0.712296
